In [1]:
import random
import time
import requests
from datetime import datetime
from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

# 크롬 드라이버 설정
DRIVER_PATH = "/Users/kim-youngho/Desktop/Sellenium/chromedriver-mac-arm64/chromedriver"

options = Options()
options.add_argument("disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
options.add_argument("--headless")
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

service = Service(DRIVER_PATH)
driver = webdriver.Chrome(service=service, options=options)


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
from bs4 import BeautifulSoup
from datetime import datetime
import time
import random

# ⭐ 설정 변수
STOP_DATE = datetime(2023, 1, 1).date()    # 이 날짜 이전 기사는 수집 안 함

driver.get("https://www.mk.co.kr/news/realestate/apartment/")
time.sleep(5)

news = []
collected_urls = set()
stop_collecting = False

def parse_articles():
    soup = BeautifulSoup(driver.page_source, 'lxml')
    ul = soup.find('ul', {'class': 'latest_news_list'})
    if not ul:
        return []

    li_list = ul.find_all('li', {'class': 'news_node'})
    articles = []

    for li in li_list:
        a_tag = li.find("a", class_="news_item")
        if a_tag:
            title_tag = a_tag.find("h3", class_="news_ttl")
            time_area = li.find("div", class_="time_area")
            time_span = time_area.find("span") if time_area else None

            raw_date = time_span.get_text(separator=" ").strip() if time_span else None
            formatted_date = None
            date_obj = None

            if raw_date:
                try:
                    # 예: "05.22 2025"
                    date_obj = datetime.strptime(raw_date, "%m.%d %Y").date()
                    formatted_date = date_obj.strftime("%Y-%m-%d")
                except ValueError:
                    formatted_date = raw_date

            if title_tag and date_obj and date_obj >= STOP_DATE:
                articles.append({
                    'url': a_tag.get("href"),
                    'title': title_tag.text.strip(),
                    'date': formatted_date,
                    'date_obj': date_obj
                })
            elif date_obj and date_obj < STOP_DATE:
                global stop_collecting
                stop_collecting = True
                break  # 오래된 기사 만나면 수집 종료

    return articles

while not stop_collecting:
    new_articles = parse_articles()

    for article in new_articles:
        if article['url'] not in collected_urls:
            news.append({
                'url': article['url'],
                'title': article['title'],
                'date': article['date']
            })
            collected_urls.add(article['url'])

    print(f"[+] 수집된 뉴스 개수: {len(news)}")

    if stop_collecting:
        print("[!] 지정 날짜 이전 기사 발견, 수집 종료")
        break

    try:
        more_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.drop_sub_news_btn'))
        )
        driver.execute_script("arguments[0].click();", more_button)
        time.sleep(random.uniform(3, 6))
    except (TimeoutException, ElementClickInterceptedException):
        print("[-] 더보기 버튼 클릭 실패 또는 기사 없음")
        break

# 결과 출력 예시
for item in news[:5]:
    print(item)

[+] 수집된 뉴스 개수: 10
[+] 수집된 뉴스 개수: 20
[+] 수집된 뉴스 개수: 30
[+] 수집된 뉴스 개수: 40
[+] 수집된 뉴스 개수: 50
[+] 수집된 뉴스 개수: 60
[+] 수집된 뉴스 개수: 70
[+] 수집된 뉴스 개수: 80
[+] 수집된 뉴스 개수: 90
[+] 수집된 뉴스 개수: 100
[+] 수집된 뉴스 개수: 110
[+] 수집된 뉴스 개수: 120
[+] 수집된 뉴스 개수: 130
[+] 수집된 뉴스 개수: 140
[+] 수집된 뉴스 개수: 150
[+] 수집된 뉴스 개수: 160
[+] 수집된 뉴스 개수: 170
[+] 수집된 뉴스 개수: 180
[+] 수집된 뉴스 개수: 190
[+] 수집된 뉴스 개수: 200
[+] 수집된 뉴스 개수: 210
[+] 수집된 뉴스 개수: 220
[+] 수집된 뉴스 개수: 230
[+] 수집된 뉴스 개수: 240
[+] 수집된 뉴스 개수: 250
[+] 수집된 뉴스 개수: 260
[+] 수집된 뉴스 개수: 270
[+] 수집된 뉴스 개수: 280
[+] 수집된 뉴스 개수: 290
[+] 수집된 뉴스 개수: 300
[+] 수집된 뉴스 개수: 310
[+] 수집된 뉴스 개수: 320
[+] 수집된 뉴스 개수: 330
[+] 수집된 뉴스 개수: 340
[+] 수집된 뉴스 개수: 350
[+] 수집된 뉴스 개수: 360
[+] 수집된 뉴스 개수: 370
[+] 수집된 뉴스 개수: 380
[+] 수집된 뉴스 개수: 390
[+] 수집된 뉴스 개수: 400
[+] 수집된 뉴스 개수: 410
[+] 수집된 뉴스 개수: 420
[+] 수집된 뉴스 개수: 430
[+] 수집된 뉴스 개수: 440
[+] 수집된 뉴스 개수: 450
[+] 수집된 뉴스 개수: 460
[+] 수집된 뉴스 개수: 470
[+] 수집된 뉴스 개수: 480
[+] 수집된 뉴스 개수: 490
[+] 수집된 뉴스 개수: 500
[+] 수집된 뉴스 개수: 510
[+] 수집된 뉴스 개수: 520
[+] 수집된 뉴스 개수: 530
[+

In [12]:
import pandas as pd
news_df = pd.DataFrame(news)

In [16]:
news_df.to_csv("../csv/daily_economy_url.csv", index=False)

In [2]:
import csv

with open('../csv/daily_economy_url.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    news = list(reader)

In [ ]:
articles = []
for idx, item in tqdm(enumerate(news), total=len(news)):
    url = item['url']
    title = item['title']  # 제목 불러오기
    try:
        driver.get(url)
        #time.sleep(1)  # 본문 페이지 로딩 대기
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # 다양한 구조 대응
        container = soup.find('div', class_='article_view') or \
                    soup.find('div', id='harmonyContainer') or \
                    soup.find('section')

        if container:
            paragraphs = container.find_all(['p', 'div'])
            texts = [p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)]
            full_text = ' '.join(texts)
            articles.append({
                'url': url,
                'title': title,  # 제목 포함
                'content': full_text
            })
        else:
            print(f"[{idx+1}] 본문 없음: {url}")
            articles.append({
                'url': url,
                'title': title,
                'content': ''
            })
    except Exception as e:
        print(f"[{idx+1}] 오류 발생: {e}")
        articles.append({
            'url': url,
            'title': title,
            'content': ''
        })

print(f"총 수집된 기사 수: {len(articles)}")

# 웹드라이버 종료
driver.quit()

  6%|▋         | 64/990 [03:39<9:32:40, 37.11s/it]

[64] 오류 발생: HTTPConnectionPool(host='localhost', port=57197): Read timed out. (read timeout=120)


  7%|▋         | 65/990 [05:39<15:55:29, 61.98s/it]

[65] 오류 발생: HTTPConnectionPool(host='localhost', port=57197): Read timed out. (read timeout=120)


  7%|▋         | 66/990 [07:39<20:22:32, 79.39s/it]

...

[735] 오류 발생: HTTPConnectionPool(host='localhost', port=57197): Read timed out. (read timeout=120)


 75%|███████▌  | 744/990 [12:14:29<2:58:01, 43.42s/it] 

[744] 오류 발생: HTTPConnectionPool(host='localhost', port=57197): Read timed out. (read timeout=120)


 75%|███████▌  | 745/990 [12:16:29<4:31:06, 66.40s/it]

[745] 오류 발생: HTTPConnectionPool(host='localhost', port=57197): Read timed out. (read timeout=120)


990